In [2]:
import pandas as pd
df = pd.read_csv(r"C:\Users\cmostajo\Documents\Dev\ProjetsWeb\catalogue_v3\data\registreDepot.csv", sep = ';', encoding = 'UTF-8')
print(df.head(10))
# On ne garde que les vraies entrées de catalogue
df = df[df["rs:class"].isin(["Collection", "Data"])].copy()


                                           dct:title    rs:class  \
0    Captation - Soufflage au chalumeau - Professeur  Collection   
1                              Vidéo d'explicitation        Data   
2                                   Vidéo exocentrée        Data   
3                                   Vidéo égocentrée        Data   
4                                   Audio vibrations        Data   
5                                      Audio ambiant        Data   
6            Pédagogie du geste - Approche cognitive  Collection   
7  Pédagogie du geste - choisir son support de mé...        Data   
8         Captation - Soufflage au chalumeau - Élève  Collection   
9                              Vidéo d'explicitation        Data   

                                         dct:license  dct:Location  \
0                                                NaN         54112   
1  Creative Commons Attribution 4.0 International...         54112   
2  Creative Commons Attribution 4.0 Inter

In [3]:
import re
from IPython.display import display  # si tu es en notebook

# Pattern W3CDTF simple : YYYY ou YYYY-MM ou YYYY-MM-DD
w3c_date_pattern = re.compile(r"^\d{4}(-\d{2}(-\d{2})?)?$")

def check_w3c_date(series):
    """
    True si la valeur respecte W3CDTF simple.
    Les NaN / chaînes vides sont considérées comme invalides
    """
    s = series.astype(str).str.strip()
    return s.map(lambda x: bool(w3c_date_pattern.match(x)))

# --- Validation dates (dct:created / dct:issued) ---

for col in ["dct:created", "dct:issued"]:
    valid_col = f"valid_{col}"
    df[valid_col] = check_w3c_date(df[col])
    print(f"{col} – lignes invalides : { (~df[valid_col]).sum() }")

invalid_dates = df[~df["valid_dct:created"] | ~df["valid_dct:issued"]]

# --- Validation du type (Collection / Data) -> rs:class ---

df["valid_rs:class"] = df["rs:class"].isin(["Collection", "Data"])
invalid_type = df[~df["valid_rs:class"]]
print(f"rs:class – lignes invalides : {len(invalid_type)}")

# --- Validation du format MIME -> dct:format ---

mime_pattern = re.compile(r"^[a-zA-Z0-9.+-]+/[a-zA-Z0-9.+-]+$")

def check_mime(series):
    """
    True si :
      - la valeur est NaN (champ vide, ex. pour les collections), ou
      - la valeur respecte le pattern MIME de base.
    """
    return series.isna() | series.astype(str).str.strip().map(
        lambda x: bool(mime_pattern.match(x))
    )

df["valid_dct:format"] = check_mime(df["dct:format"])
invalid_format = df[~df["valid_dct:format"]]
print(f"dct:format – lignes invalides : {len(invalid_format)}")

# --- Résumé ---

has_errors = not (invalid_dates.empty and invalid_type.empty and invalid_format.empty)

if has_errors:
    print("\nDes erreurs de validation ont été détectées.\n")

    if not invalid_dates.empty:
        print("Dates invalides (dct:created / dct:issued) :")
        display(invalid_dates[["dct:identifier", "dct:created", "dct:issued", "rs:class"]])

    if not invalid_type.empty:
        print("\nTypes invalides (rs:class) :")
        display(invalid_type[["dct:identifier", "rs:class"]])

    if not invalid_format.empty:
        print("\nFormats MIME invalides (dct:format) :")
        display(invalid_format[["dct:identifier", "dct:format"]])
else:
    print("Validation OK : dates, type et format sont conformes.")

df = df.drop('valid_dct:format', axis=1)
df = df.drop('valid_rs:class', axis=1)
df = df.drop('valid_dct:created', axis=1)
df = df.drop('valid_dct:issued', axis=1)

dct:created – lignes invalides : 0
dct:issued – lignes invalides : 0
rs:class – lignes invalides : 0
dct:format – lignes invalides : 0
Validation OK : dates, type et format sont conformes.


In [4]:
print(df.columns)

Index(['dct:title', 'rs:class', 'dct:license', 'dct:Location', 'dct:created',
       'dct:creator', 'dct:description', 'dct:issued', 'dct:language',
       'dct:extent', 'dct:publisher', 'dct:subject', 'rs:mediatype',
       'dct:format', 'rs:storage', 'dct:identifier', 'rs:rightsHolder',
       'rs:CollectionID', 'rs:DOI', 'rs:category', 'rs:discipline',
       'rs:expertise', 'rs:audience', 'rs:subtype'],
      dtype='object')


In [5]:
import json

collections_df = df[df["rs:class"] == "Collection"].copy()
data_df = df[df["rs:class"] == "Data"].copy()

# Champs à valeurs multiples (colonnes du DataFrame)
MULTI_VALUE_COLS = [
    "dct:creator",
    "rs:rightsHolder",
    "dct:subject",
    "rs:expertise",
    "rs:audience"
]

def row_to_clean_dict(row):
    """
    Convertit une ligne en dict :
      - supprime les NaN
      - pour certains champs (MULTI_VALUE_COLS), transforme
        'A; B; C' -> ['A', 'B', 'C']
    """
    d = row.dropna().to_dict()

    for col in MULTI_VALUE_COLS:
        if col in d:
            raw = str(d[col]).strip()
            if not raw:
                # valeur vide → on retire la clé
                d.pop(col, None)
                continue

            parts = [p.strip() for p in raw.split(";") if p.strip()]
            if parts:
                d[col] = parts
            else:
                d.pop(col, None)

    return d

# Dictionnaire temporaire pour construire la hiérarchie
collections = {}

# -----------------------------
# 2. Initialiser les collections
# -----------------------------
for _, row in collections_df.iterrows():
    col_id = row["dct:identifier"]  # ou "rs:identifier" selon ton registre

    # convertir la ligne collection → dict propre (avec listes pour multi-valeurs)
    col_dict = row_to_clean_dict(row)
    col_dict["data"] = []          # ajouter la liste vide
    collections[col_id] = col_dict

# -----------------------------
# 3. Attacher chaque Data à sa Collection
# -----------------------------
for _, row in data_df.iterrows():
    parent_id = row["rs:CollectionID"]

    if parent_id in collections:
        # convertir la ligne data → dict propre (avec listes pour multi-valeurs)
        data_dict = row_to_clean_dict(row)
        collections[parent_id]["data"].append(data_dict)

# -----------------------------
# 4. Construire la structure finale
# -----------------------------
catalogue_json = {
    "collections": list(collections.values()),

    "@context": {
        "dct": "http://purl.org/dc/terms/",
        "rs": {}
    }
}

# -----------------------------
# 5. Export JSON (joli format)
# -----------------------------
with open("catalogue_hierarchique.json", "w", encoding="utf-8") as f:
    json.dump(catalogue_json, f, indent=2, ensure_ascii=False)

catalogue_json


{'collections': [{'dct:title': 'Captation - Soufflage au chalumeau - Professeur',
   'rs:class': 'Collection',
   'dct:Location': 54112,
   'dct:created': '2024-06-21',
   'dct:creator': ['David Arnaud', 'Alina Glushkova'],
   'dct:description': "Captation réalisée par les Mines Paris auprès d'Antoine Dupont, enseignant en soufflage de verre au chalumeau au CERFAV.\nContenu :\n• 2 vidéos : caméra embarquée par le sujet (égocentré) et point de vue fixe externe (égocentré)\n• 2 audios : audio ambiant et vibrations de l'établi\n• 1 vidéo d'élicitation : le sujet commente la vidéo au pdv égocentré\n• un fichier des données de capteurs motion capture ",
   'dct:issued': '2025-10-30',
   'dct:language': 'français (fr)',
   'dct:publisher': 'Re-Source France 2030',
   'dct:subject': ['verre soufflé',
    'capteurs',
    "artisanat d'art",
    'Re-Source'],
   'rs:storage': 'Nakala',
   'dct:identifier': '10.34847/nkl.f5587q9i',
   'rs:rightsHolder': ['Mines Paristech', 'CERFAV'],
   'rs:categ